# bioseqkits FASTA 分析 Notebook

这个 Notebook 演示如何直接加载 `src/bioseqkits` 模块，解析 `tests/data/sequence.fasta`，并输出统计结果。

### 1.设置工作路径并将 `src` 添加到 `sys.path`，确保 Notebook 能直接导入 `bioseqkits` 包。

预计输出:
- `project_root = ...`
- `src_path = ...`
- `sys.path[0] = ...`

In [ ]:
import sys
from pathlib import Path

# 显式查找项目根目录并将 src 添加到 sys.path
project_root = Path.cwd()
src_path = project_root / 'src'
while not src_path.exists() and project_root != project_root.parent:
    project_root = project_root.parent
    src_path = project_root / 'src'
assert src_path.exists(), '请从项目根目录运行 Notebook，且项目目录必须包含 src 文件夹'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 预计输出:
# project_root = <project_root_path>
# src_path = <project_root>/src
# sys.path[0] = <project_root>/src
print('project_root =', project_root)
print('src_path =', src_path)
print('sys.path[0] =', sys.path[0])

### 2.导入 `bioseqkits` 核心模块，并验证包加载是否正常。

预计输出:
- `imports succeeded`

In [ ]:
from bioseqkits.utils import open_sequence_file, detect_format, FileFormat
from bioseqkits.parser import parse_fasta
from bioseqkits.stats import calculate_sequence_stats
from bioseqkits.operations import (
    reverse_complement,
    dna_to_rna,
    rna_to_dna,
    translate,
    six_frame_translation,
    kmer_frequency,
    top_k_kmers,
)

# 预计输出:
# imports succeeded
print('imports succeeded')

### 3.加载 FASTA 样例文件，检测格式并解析记录，检查解析是否正确。

预计输出:
- `fasta_path exists: True`
- `fasta_path: ...`
- `detected format: FileFormat.FASTA`
- `detected format value: fasta`
- `record count = 1`
- 第一条记录的 ID、长度和前 80 个碱基

In [ ]:
fasta_path = Path('tests') / 'data' / 'sequence.fasta'
# 预计输出:
# fasta_path exists: True
# fasta_path: tests/data/sequence.fasta
print('fasta_path exists:', fasta_path.exists())
print('fasta_path:', fasta_path)

file_format = detect_format(fasta_path)
# 预计输出:
# detected format: FileFormat.FASTA
# detected format value: fasta
print('detected format:', file_format)
print('detected format value:', getattr(file_format, 'value', file_format))

records = []
with open_sequence_file(fasta_path) as fh:
    for record in parse_fasta(fh):
        records.append(record)

# 预计输出:
# record count = 1
# first record id = ...
# first record seq length = ...
# first 80 bases = ...
print('record count =', len(records))
if records:
    print('first record id =', records[0].id)
    print('first record seq length =', len(records[0].seq))
    print('first 80 bases =', records[0].seq[:80])

### 4.计算序列统计信息：长度、GC 含量、N 比例和碱基组成。

预计输出:
- `statistics for first record:`
- `length: ...`
- `gc_content: ...`
- `n_ratio: ...`
- `A: ...`
- `C: ...`
- `G: ...`
- `T: ...`
- `N: ...`

In [ ]:
if records:
    stats = calculate_sequence_stats(records[0])
    # 预计输出:
    # statistics for first record:
    #   length: ...
    #   gc_content: ...
    #   n_ratio: ...
    #   A: ...
    #   C: ...
    #   G: ...
    #   T: ...
    #   N: ...
    print('statistics for first record:')
    for key, value in stats.items():
        print(f'  {key}: {value}')

### 5.测试反向互补、DNA/RNA 转换、六框翻译和 k-mer 频数统计。

预计输出:
- `reverse complement: ...`
- `dna -> rna: ...`
- `rna -> dna: ...`
- `six frame translation:` 6 行翻译结果
- `kmer frequency:` 字典输出
- `top k-mers:` 列表输出

In [ ]:
if records:
    seq = records[0].seq
    print('reverse complement:', reverse_complement(seq))
    print('dna -> rna:', dna_to_rna(seq[:50]))
    print('rna -> dna:', rna_to_dna(dna_to_rna(seq[:50])))

    proteins = six_frame_translation(seq[:60])
    print('six frame translation:')
    for i, protein in enumerate(proteins, start=1):
        print(f'  frame {i}:', protein)

    kmer_freq = kmer_frequency(seq, k=3)
    print('kmer frequency sample:', dict(list(kmer_freq.items())[:10]))
    print('top k-mers:', top_k_kmers(kmer_freq, top_k=5))